In [0]:
print(spark)
print("Spark is ready!")

Spark is ready!


In [0]:
#Create data
data = [(1,), (2,), (3,), (4,), (5,), (6,), (7,), (8,)]

df = spark.createDataFrame(data, ["number"])

display(df)

number
1
2
3
4
5
6
7
8


In [0]:
from pyspark.sql.functions import col

result = df.select((col("number") ** 2).alias("square"))

display(result)

square
1.0
4.0
9.0
16.0
25.0
36.0
49.0
64.0


In [0]:
display(df.filter(col("number") % 2 == 0))

number
2
4
6
8


In [0]:
from pyspark.sql.functions import sum

sales = [
    ("Bangalore", 1000),
    ("Chennai", 1500),
    ("Bangalore", 2000),
    ("Chennai", 500),
    ("Hyderabad", 1200)
]

sales_df = spark.createDataFrame(sales, ["city", "amount"])

result = sales_df.groupBy("city").agg(
    sum("amount").alias("total_sales")
)

display(result)

city,total_sales
Bangalore,3000
Chennai,2000
Hyderabad,1200


In [0]:
data = [
    ("Apple", 10),
    ("Mango", 20),
    ("Apple", 15),
    ("Banana", 8),
    ("Mango", 5),
    ("Apple", 25)
]

df = spark.createDataFrame(data, ["fruit", "quantity"])

display(df)

fruit,quantity
Apple,10
Mango,20
Apple,15
Banana,8
Mango,5
Apple,25


In [0]:
mapped_df = df.selectExpr(
    "fruit",
    "quantity",
    "quantity * 2 AS doubled_quantity"
)

display(mapped_df)

fruit,quantity,doubled_quantity
Apple,10,20
Mango,20,40
Apple,15,30
Banana,8,16
Mango,5,10
Apple,25,50


In [0]:
#flatMap() — split sentences into words
from pyspark.sql.functions import explode, split, col, lower

sentences = [
    ("Spark is fast",),
    ("Spark processes data",),
    ("Data is useful",)
]

text_df = spark.createDataFrame(sentences, ["sentence"])

words_df = (
    text_df
    .select(explode(split(lower(col("sentence")), " ")).alias("word"))
)

display(words_df)

word
spark
is
fast
spark
processes
data
data
is
useful


In [0]:
filtered_df = df.filter(col("fruit") == "Apple")

display(filtered_df)

fruit,quantity
Apple,10
Apple,15
Apple,25


In [0]:
display(df.select("fruit").distinct())

fruit
Apple
Mango
Banana


In [0]:
df1 = spark.createDataFrame(
    [("Apple", 10), ("Mango", 20)],
    ["fruit", "quantity"]
)

df2 = spark.createDataFrame(
    [("Banana", 8), ("Apple", 15)],
    ["fruit", "quantity"]
)

combined_df = df1.union(df2)

display(combined_df)

fruit,quantity
Apple,10
Mango,20
Banana,8
Apple,15


In [0]:
#Actions
#Actions return results or trigger execution of the Spark query plan.
#collect() — retrieve all rows
rows = df.collect()

for row in rows:
    print(row)

Row(fruit='Apple', quantity=10)
Row(fruit='Mango', quantity=20)
Row(fruit='Apple', quantity=15)
Row(fruit='Banana', quantity=8)
Row(fruit='Mango', quantity=5)
Row(fruit='Apple', quantity=25)


In [0]:
print(df.first())

Row(fruit='Apple', quantity=10)


In [0]:
for row in df.take(3):
    print(row)

Row(fruit='Apple', quantity=10)
Row(fruit='Mango', quantity=20)
Row(fruit='Apple', quantity=15)


In [0]:
df.show()

+------+--------+
| fruit|quantity|
+------+--------+
| Apple|      10|
| Mango|      20|
| Apple|      15|
|Banana|       8|
| Mango|       5|
| Apple|      25|
+------+--------+



In [0]:
from pyspark.sql.functions import sum

df.select(sum("quantity").alias("total_quantity")).show()

+--------------+
|total_quantity|
+--------------+
|            83|
+--------------+



In [0]:
#5. Grouping operations
# groupBy() — group fruits and count records
from pyspark.sql.functions import count

result = df.groupBy("fruit").agg(
    count("*").alias("fruit_count")
)

display(result)

fruit,fruit_count
Apple,3
Mango,2
Banana,1


In [0]:
result = df.groupBy("fruit").agg(
    sum("quantity").alias("total_quantity")
)

display(result)

fruit,total_quantity
Apple,50
Mango,25
Banana,8


In [0]:
#groupByKey() equivalent — collect values by key
#In Pair RDDs, groupByKey() groups all values sharing a key. The following DataFrame operation produces equivalent grouped values.
from pyspark.sql.functions import collect_list

result = df.groupBy("fruit").agg(
    collect_list("quantity").alias("quantities")
)

display(result)

fruit,quantities
Apple,"List(10, 15, 25)"
Mango,"List(20, 5)"
Banana,List(8)


In [0]:
#countByKey() equivalent — count each key
result = df.groupBy("fruit").count()

display(result)

fruit,count
Apple,3
Mango,2
Banana,1


In [0]:
#reduceByKey() equivalent — add values for each key
result = df.groupBy("fruit").agg(
    sum("quantity").alias("total")
)

display(result)

fruit,total
Apple,50
Mango,25
Banana,8


In [0]:
prices = spark.createDataFrame(
    [("Apple", 100), ("Mango", 80), ("Banana", 40)],
    ["fruit", "price"]
)

joined_df = df.join(prices, on="fruit", how="inner")

display(joined_df)

fruit,quantity,price
Apple,10,100
Mango,20,80
Apple,15,100
Banana,8,40
Mango,5,80
Apple,25,100


In [0]:
display(df.orderBy(col("quantity").desc()))

fruit,quantity
Apple,25
Mango,20
Apple,15
Apple,10
Banana,8
Mango,5


In [0]:
result = df.groupBy("fruit").agg(
    sum("quantity").alias("total_quantity")
)

display(result.orderBy(col("total_quantity").desc()))

fruit,total_quantity
Apple,50
Mango,25
Banana,8


In [0]:
from pyspark.sql.functions import sum, count, avg

summary = df.groupBy("fruit").agg(
    count("*").alias("number_of_records"),
    sum("quantity").alias("total_quantity"),
    avg("quantity").alias("average_quantity")
)

display(summary.orderBy("fruit"))

fruit,number_of_records,total_quantity,average_quantity
Apple,3,50,16.666666666666668
Banana,1,8,8.0
Mango,2,25,12.5
